In [3]:
from pathlib import Path
import time

import pandas as pd
import numpy as np
import requests

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

import geopandas as gpd
import osmnx as ox

In [4]:
BASE_DIR = Path("data/external")

WALMART_DIR = BASE_DIR / "walmart"
OSM_DIR = BASE_DIR / "osm"
STATCAN_DIR = BASE_DIR / "statscan"
WEATHER_DIR = BASE_DIR / "weather"
HOLIDAY_DIR = BASE_DIR / "holidays"

for folder in [
    WALMART_DIR,
    OSM_DIR,
    STATCAN_DIR,
    WEATHER_DIR,
    HOLIDAY_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

In [5]:
stores = pd.DataFrame([
    ["3106", "Toronto Dufferin Mall",
     "900 Dufferin St", "Toronto", "ON", "M6H 4A9"],

    ["1139", "Toronto Downsview",
     "3757 Keele St", "Toronto", "ON", "M3J 1N4"],

    ["1004", "Toronto Stockyards",
     "2525 St Clair Ave W", "Toronto", "ON", "M6N 4Z5"],

    ["1150", "Toronto Downtown East",
     "1000 Gerrard St E", "Toronto", "ON", "M4M 3G6"],

    ["3105", "Toronto North Park",
     "1305 Lawrence Ave W", "Toronto", "ON", "M6L 1A5"],

    ["3031", "Etobicoke North Queen",
     "165 N Queen St", "Etobicoke", "ON", "M9C 1A7"],

    ["3055", "Mississauga Square One",
     "100 City Centre Dr", "Mississauga", "ON", "L5B 2C9"],

    ["1061", "Mississauga Heartland",
     "800 Matheson Blvd W", "Mississauga", "ON", "L5V 2N6"],

    ["1126", "Mississauga East",
     "1500 Dundas St E", "Mississauga", "ON", "L4X 1L4"],

    ["3054", "Mississauga Meadowvale",
     "3155 Argentia Rd", "Mississauga", "ON", "L5N 8E1"],

    ["PORT_CREDIT", "Mississauga Port Credit",
     "515 Lakeshore Rd E", "Mississauga", "ON", "L5G 1H9"],

    ["1079", "Brampton West",
     "9455 Mississauga Rd", "Brampton", "ON", "L6X 0Z8"],

    ["1188", "Brampton South",
     "15 Resolution Dr", "Brampton", "ON", "L6W 0A6"],

    ["3135", "Brampton East",
     "30 Coventry Rd", "Brampton", "ON", "L6T 5P9"],

    ["3130", "Brampton North",
     "50 Quarry Edge Dr", "Brampton", "ON", "L6V 4K2"]
],
columns=[
    "store_id",
    "store_name",
    "address",
    "city",
    "province",
    "postal_code"
])

stores

,store_id,store_name,address,city,province,postal_code
0,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9
1,1139,Toronto Downsview,3757 Keele St,Toronto,ON,M3J 1N4
2,1004,Toronto Stockyards,2525 St Clair Ave W,Toronto,ON,M6N 4Z5
3,1150,Toronto Downtown East,1000 Gerrard St E,Toronto,ON,M4M 3G6
4,3105,Toronto North Park,1305 Lawrence Ave W,Toronto,ON,M6L 1A5
5,3031,Etobicoke North Queen,165 N Queen St,Etobicoke,ON,M9C 1A7
6,3055,Mississauga Square One,100 City Centre Dr,Mississauga,ON,L5B 2C9
7,1061,Mississauga Heartland,800 Matheson Blvd W,Mississauga,ON,L5V 2N6
8,1126,Mississauga East,1500 Dundas St E,Mississauga,ON,L4X 1L4
9,3054,Mississauga Meadowvale,3155 Argentia Rd,Mississauga,ON,L5N 8E1


In [6]:
stores["full_address"] = (
    stores["address"] + ", " +
    stores["city"] + ", " +
    stores["province"] + " " +
    stores["postal_code"] +
    ", Canada"
)

stores[
    ["store_id", "store_name", "full_address"]
].head()

,store_id,store_name,full_address
0,3106,Toronto Dufferin Mall,"900 Dufferin St, Toronto, ON M6H 4A9, Canada"
1,1139,Toronto Downsview,"3757 Keele St, Toronto, ON M3J 1N4, Canada"
2,1004,Toronto Stockyards,"2525 St Clair Ave W, Toronto, ON M6N 4Z5, Canada"
3,1150,Toronto Downtown East,"1000 Gerrard St E, Toronto, ON M4M 3G6, Canada"
4,3105,Toronto North Park,"1305 Lawrence Ave W, Toronto, ON M6L 1A5, Canada"


In [7]:
stores.to_parquet(
    WALMART_DIR / "gta_walmart_stores_raw.parquet",
    index=False
)

In [8]:
geolocator = Nominatim(
    user_agent="retail-promotion-intelligence"
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1
)

In [9]:
def get_coordinates(address):
    try:
        location = geocode(address)

        if location is None:
            return pd.Series([np.nan, np.nan])

        return pd.Series([
            location.latitude,
            location.longitude
        ])

    except Exception as e:
        print("Geocoding error:", e)
        return pd.Series([np.nan, np.nan])

In [10]:
stores[
    ["latitude", "longitude"]
] = stores["full_address"].apply(
    get_coordinates
)

stores[
    [
        "store_name",
        "latitude",
        "longitude"
    ]
]

,store_name,latitude,longitude
0,Toronto Dufferin Mall,43.655980,-79.435728
1,Toronto Downsview,43.757509,-79.489316
2,Toronto Stockyards,43.667987,-79.484753
3,Toronto Downtown East,43.669300,-79.339861
4,Toronto North Park,43.708480,-79.473550
5,Etobicoke North Queen,NaN,NaN
6,Mississauga Square One,43.592174,-79.640972
7,Mississauga Heartland,43.607226,-79.693799
8,Mississauga East,43.608997,-79.577470
9,Mississauga Meadowvale,43.594297,-79.789161


In [11]:
failed = stores[
    stores["latitude"].isna()
    | stores["longitude"].isna()
]

print("Failed:", len(failed))

failed

Failed: 1


,store_id,store_name,address,city,province,postal_code,full_address,latitude,longitude
5,3031,Etobicoke North Queen,165 N Queen St,Etobicoke,ON,M9C 1A7,"165 N Queen St, Etobicoke, ON M9C 1A7, Canada",NaN,NaN


In [12]:
stores.to_parquet(
    WALMART_DIR / "gta_walmart_stores_geocoded.parquet",
    index=False
)

In [13]:
RADIUS_M = 5000

In [14]:
OSM_TAGS = {
    "amenity": [
        "university",
        "college",
        "school",
        "bus_station"
    ],
    "office": True,
    "landuse": [
        "industrial",
        "commercial",
        "retail"
    ],
    "railway": [
        "station",
        "halt"
    ]
}

In [15]:
def collect_osm_features(row):

    point = (
        row["latitude"],
        row["longitude"]
    )

    try:

        gdf = ox.features_from_point(
            point,
            tags=OSM_TAGS,
            dist=RADIUS_M
        )

        return gdf

    except Exception as e:
        print(
            row["store_name"],
            "OSM error:",
            e
        )

        return gpd.GeoDataFrame()

In [16]:
osm_results = []

for _, store in stores.iterrows():

    print("Processing:", store["store_name"])

    features_gdf = collect_osm_features(store)

    if features_gdf.empty:

        osm_results.append({
            "store_id": store["store_id"],
            "universities": 0,
            "colleges": 0,
            "schools": 0,
            "offices": 0,
            "industrial": 0,
            "commercial": 0,
            "retail_landuse": 0,
            "rail_stations": 0
        })

        continue

    amenity = (
        features_gdf["amenity"]
        if "amenity" in features_gdf.columns
        else pd.Series(index=features_gdf.index, dtype=object)
    )

    landuse = (
        features_gdf["landuse"]
        if "landuse" in features_gdf.columns
        else pd.Series(index=features_gdf.index, dtype=object)
    )

    railway = (
        features_gdf["railway"]
        if "railway" in features_gdf.columns
        else pd.Series(index=features_gdf.index, dtype=object)
    )

    office = (
        features_gdf["office"]
        if "office" in features_gdf.columns
        else pd.Series(index=features_gdf.index, dtype=object)
    )

    osm_results.append({

        "store_id": store["store_id"],

        "universities":
            (amenity == "university").sum(),

        "colleges":
            (amenity == "college").sum(),

        "schools":
            (amenity == "school").sum(),

        "offices":
            office.notna().sum(),

        "industrial":
            (landuse == "industrial").sum(),

        "commercial":
            (landuse == "commercial").sum(),

        "retail_landuse":
            (landuse == "retail").sum(),

        "rail_stations":
            railway.isin(
                ["station", "halt"]
            ).sum()
    })

    time.sleep(1)

Processing: Toronto Dufferin Mall
Processing: Toronto Downsview
Processing: Toronto Stockyards
Processing: Toronto Downtown East
Processing: Toronto North Park
Processing: Etobicoke North Queen
Etobicoke North Queen OSM error: IllegalArgumentException: Points of LinearRing do not form a closed linestring
Processing: Mississauga Square One
Processing: Mississauga Heartland
Processing: Mississauga East
Processing: Mississauga Meadowvale
Processing: Mississauga Port Credit
Processing: Brampton West
Processing: Brampton South
Processing: Brampton East
Processing: Brampton North


In [17]:
osm_context = pd.DataFrame(osm_results)

osm_context

,store_id,universities,colleges,schools,offices,industrial,commercial,retail_landuse,rail_stations
0,3106,19,19,206,940,109,172,519,44
1,1139,3,8,112,245,159,82,216,18
2,1004,0,4,171,653,132,145,498,24
3,1150,20,22,178,757,77,108,424,36
4,3105,1,5,176,360,130,128,427,17
5,3031,0,0,0,0,0,0,0,0
6,3055,2,19,107,274,92,103,235,3
7,1061,0,8,84,227,115,89,138,3
8,1126,0,17,106,509,108,146,336,6
9,3054,0,0,40,47,94,46,62,2


In [18]:
osm_context.to_parquet(
    OSM_DIR / "store_osm_context_5km.parquet",
    index=False
)

In [19]:
demographic_columns = [
    "store_id",

    "population",
    "population_density",

    "median_household_income",
    "average_household_size",

    "age_15_24_pct",
    "age_25_44_pct",
    "age_45_64_pct",

    "renter_pct",

    "postsecondary_pct",

    "employment_rate",

    "manufacturing_workers_pct",
    "transport_warehouse_workers_pct",
    "professional_workers_pct"
]

demographic_template = pd.DataFrame(
    columns=demographic_columns
)

demographic_template

,store_id,population,population_density,median_household_income,average_household_size,age_15_24_pct,age_25_44_pct,age_45_64_pct,renter_pct,postsecondary_pct,employment_rate,manufacturing_workers_pct,transport_warehouse_workers_pct,professional_workers_pct


In [20]:
demographic_template.to_parquet(
    STATCAN_DIR / "store_demographics_template.parquet",
    index=False
)

In [21]:
weather_columns = [
    "date",
    "station_id",
    "mean_temperature",
    "min_temperature",
    "max_temperature",
    "total_rain",
    "total_snow",
    "total_precipitation"
]

weather_template = pd.DataFrame(
    columns=weather_columns
)

weather_template.to_parquet(
    WEATHER_DIR / "weather_template.parquet",
    index=False
)

In [22]:
%pip install holidays

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [23]:
import holidays

In [24]:
ontario_holidays = holidays.CA(
    subdiv="ON",
    years=range(2011, 2027)
)

In [25]:
holiday_df = pd.DataFrame(
    [
        {
            "date": pd.Timestamp(date),
            "holiday_name": name,
            "is_holiday": 1
        }
        for date, name in ontario_holidays.items()
    ]
)

holiday_df = holiday_df.sort_values("date")

holiday_df.head()

,date,holiday_name,is_holiday
106,2011-01-01,New Year's Day,1
107,2011-01-03,New Year's Day (observed),1
113,2011-02-21,Family Day,1
108,2011-04-22,Good Friday,1
114,2011-05-23,Victoria Day,1


In [26]:
holiday_df.to_parquet(
    HOLIDAY_DIR / "ontario_holidays.parquet",
    index=False
)

In [27]:
store_context = stores.merge(
    osm_context,
    on="store_id",
    how="left",
    validate="one_to_one"
)

store_context.head()

,store_id,store_name,address,city,province,postal_code,full_address,latitude,longitude,universities,colleges,schools,offices,industrial,commercial,retail_landuse,rail_stations
0,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.655980,-79.435728,19,19,206,940,109,172,519,44
1,1139,Toronto Downsview,3757 Keele St,Toronto,ON,M3J 1N4,"3757 Keele St, Toronto, ON M3J 1N4, Canada",43.757509,-79.489316,3,8,112,245,159,82,216,18
2,1004,Toronto Stockyards,2525 St Clair Ave W,Toronto,ON,M6N 4Z5,"2525 St Clair Ave W, Toronto, ON M6N 4Z5, Canada",43.667987,-79.484753,0,4,171,653,132,145,498,24
3,1150,Toronto Downtown East,1000 Gerrard St E,Toronto,ON,M4M 3G6,"1000 Gerrard St E, Toronto, ON M4M 3G6, Canada",43.669300,-79.339861,20,22,178,757,77,108,424,36
4,3105,Toronto North Park,1305 Lawrence Ave W,Toronto,ON,M6L 1A5,"1305 Lawrence Ave W, Toronto, ON M6L 1A5, Canada",43.708480,-79.473550,1,5,176,360,130,128,427,17


In [28]:
print(
    "Stores:",
    store_context["store_id"].nunique()
)

print(
    "Duplicate stores:",
    store_context["store_id"].duplicated().sum()
)

print(
    "Missing latitude:",
    store_context["latitude"].isna().sum()
)

print(
    "Missing longitude:",
    store_context["longitude"].isna().sum()
)

Stores: 15
Duplicate stores: 0
Missing latitude: 1
Missing longitude: 1


In [29]:
store_context[
    [
        "store_name",
        "universities",
        "colleges",
        "schools",
        "offices",
        "industrial",
        "commercial",
        "rail_stations"
    ]
].sort_values(
    "universities",
    ascending=False
)

,store_name,universities,colleges,schools,offices,industrial,commercial,rail_stations
3,Toronto Downtown East,20,22,178,757,77,108,36
0,Toronto Dufferin Mall,19,19,206,940,109,172,44
12,Brampton South,4,14,79,159,166,52,2
1,Toronto Downsview,3,8,112,245,159,82,18
11,Brampton West,3,4,69,50,28,29,2
14,Brampton North,3,7,116,107,50,28,2
6,Mississauga Square One,2,19,107,274,92,103,3
4,Toronto North Park,1,5,176,360,130,128,17
13,Brampton East,1,3,79,95,160,32,2
2,Toronto Stockyards,0,4,171,653,132,145,24


In [30]:
store_context.to_parquet(
    BASE_DIR / "gta_store_context_v1.parquet",
    index=False
)

print("Saved.")

Saved.


In [31]:
print("08 EXTERNAL DATA COLLECTION SUMMARY")
print("-----------------------------------")

print(
    "Walmart stores:",
    len(stores)
)

print(
    "Geocoded stores:",
    stores["latitude"].notna().sum()
)

print(
    "OSM store contexts:",
    len(osm_context)
)

print(
    "Holiday records:",
    len(holiday_df)
)

08 EXTERNAL DATA COLLECTION SUMMARY
-----------------------------------
Walmart stores: 15
Geocoded stores: 14
OSM store contexts: 15
Holiday records: 154
